# 원-핫 인코딩

원-핫 인코딩은 토큰의 정수 ID 위치만 1, 나머지는 0인 벡터로 바꾸는 방법이다. 정수 ID를 연속형 값처럼 직접 쓰는 오류를 피하며, 임베딩에서 ID를 조회용으로 쓰는 것과는 다르다. 단어 집합이 커질수록 벡터가 길고 희소해지므로 의미 관계가 필요하면 임베딩을 사용한다.

### NLTK 리소스 준비

토큰화와 영어 불용어 처리에 사용할 NLTK 리소스를 내려받는다.


In [1]:
import nltk
from tensorflow.python.ops import nn

nltk.download("punkt")
nltk.download("punkt_tab")

nltk.download("stopwords")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 01. 원-핫 인코딩
정수 ID는 토큰을 구별하기 위한 번호일 뿐 크기나 순서를 뜻하지 않는다.
ID를 실수 특성처럼 직접 넣으면 모델은 ID 6이 ID 1보다 크고 두 값의 거리가 5라는 잘못된 관계를 학습할 수 있다.

**원-핫 인코딩**은 ID를 값이 아니라 1의 위치로 사용해 숫자 크기 해석을 막는다. 단어 집합 크기가 $V$이면 토큰 하나는 길이 $V$인 벡터가 되고, 한 문장은 `(토큰 수, V)`, 여러 문장은 `(문장 수, 문장 길이, V)` 형태가 된다.

원-핫 벡터는 단어 사이의 의미 유사성을 표현하지 못하므로 의미 관계가 필요하면 임베딩을 사용한다.

- **단어 집합(Vocabulary)**: 서로 구분할 고유 토큰의 모음
- **정수 인코딩(Integer Encoding)**: 단어 집합의 각 토큰에 고유한 정수 ID를 부여하는 과정
- **원-핫 인코딩(One-Hot Encoding)**: 단어 집합 크기의 벡터에서 토큰 ID 위치만 1로 표시하는 방법
- **희소 벡터(Sparse Vector)**: 대부분의 원소가 0인 벡터
- **임베딩(Embedding)**: 토큰을 짧은 실수 벡터로 바꾸고 학습을 통해 의미 관계를 표현하는 방법



### 토큰화로 입력 단위 준비

문자열을 토큰 목록으로 나누면 각 토큰 순서가 정수 ID와 원-핫 행렬의 행 순서로 이어진다.


In [2]:
from nltk.tokenize import word_tokenize

sentence = "I love NLP so much and I love you"

tokens = word_tokenize(sentence)
print("토큰:", tokens)
print("토큰 수:", len(tokens))


토큰: ['I', 'love', 'NLP', 'so', 'much', 'and', 'I', 'love', 'you']
토큰 수: 9


### 단어 집합과 정수 인코딩

고유 토큰의 빈도를 센 뒤 각 토큰에 0부터 ID를 부여한다. 각 토큰을 사전에서 조회하면 원래 순서의 정수 ID 목록이 된다.

- **`Counter()`**: 토큰별 등장 횟수를 세어 `{토큰: 빈도}` 형태로 반환하는 클래스
- **`enumerate()`**: 반복 가능한 값에 0부터 시작하는 순번을 함께 제공하는 함수

`Counter()`가 만든 고유 토큰 순서에 `enumerate()`의 번호를 연결하면 토큰을 ID로 바꾸는 사전이 완성된다.


In [3]:
from collections import Counter

word_counts = Counter(tokens)

word_to_index = {
    word: index
    for index, word in enumerate(word_counts)
}

sequence = [
    word_to_index[token]
    for token in tokens
]

print("토큰 빈도:", word_counts)
print("단어 → ID:", word_to_index)
print("정수 인코딩:", sequence)


토큰 빈도: Counter({'I': 2, 'love': 2, 'NLP': 1, 'so': 1, 'much': 1, 'and': 1, 'you': 1})
단어 → ID: {'I': 0, 'love': 1, 'NLP': 2, 'so': 3, 'much': 4, 'and': 5, 'you': 6}
정수 인코딩: [0, 1, 2, 3, 4, 5, 0, 1, 6]


### NumPy로 원-핫 벡터 만들기

먼저 `(토큰 수, 단어 집합 크기)`의 0 행렬을 만들고 각 행의 토큰 ID 열만 1로 바꾼다. 행은 토큰 위치, 열은 단어 종류를 나타낸다.

- **`np.zeros()`**: 지정한 shape의 모든 원소를 0으로 채운 NumPy 배열을 만드는 함수
- **`one_hot_encode()`**: 정수 ID 목록을 받아 각 ID 위치만 1인 원-핫 행렬로 바꾸는 사용자 정의 함수
- **`zip()`**: 여러 반복 가능한 값에서 같은 위치의 원소를 묶어 함께 순회하는 함수

정수 ID가 열 번호를 선택하므로 출력에서 `ID`와 값이 1인 열이 일치하는지 비교하면 변환이 올바른지 확인할 수 있다. 이 예제의 단어 집합 크기는 7이므로 `NLP`의 ID 2는 길이 7의 `[0, 0, 1, 0, 0, 0, 0]`으로 바뀐다.


In [4]:
import numpy as np

vocab_size = len(word_to_index)  # 7

def one_hot_encode(token_ids, vector_size):
    """정수 ID 목록을 (토큰 수, 단어 집합 크기) 원-핫 행렬로 변환한다."""

    vectors = np.zeros((len(token_ids), vector_size), dtype=np.int64)

    # 9행 7열 vectors 표에서
    # 각 행에서 token_id가 일치하는 컬럼만 1로 변경
    for row_index, token_id in enumerate(token_ids):
        vectors[row_index, token_id] = 1

    return vectors

one_hot_vectors = one_hot_encode(sequence, vocab_size)

print("행렬 shape:", one_hot_vectors.shape)
for token, token_id, vector in zip(tokens, sequence, one_hot_vectors):
    print(f"{token:>4} | ID={token_id} | {vector.tolist()}")


행렬 shape: (9, 7)
   I | ID=0 | [1, 0, 0, 0, 0, 0, 0]
love | ID=1 | [0, 1, 0, 0, 0, 0, 0]
 NLP | ID=2 | [0, 0, 1, 0, 0, 0, 0]
  so | ID=3 | [0, 0, 0, 1, 0, 0, 0]
much | ID=4 | [0, 0, 0, 0, 1, 0, 0]
 and | ID=5 | [0, 0, 0, 0, 0, 1, 0]
   I | ID=0 | [1, 0, 0, 0, 0, 0, 0]
love | ID=1 | [0, 1, 0, 0, 0, 0, 0]
 you | ID=6 | [0, 0, 0, 0, 0, 0, 1]


### 희소성과 단어 간 거리

희소성은 전체 원소 중 0의 비율로 확인한다. 서로 다른 원-핫 벡터의 유클리드 거리는 항상 $\sqrt{2}$이므로 의미 관계를 구분하지 못한다.

- **희소성(Sparsity)**: 배열에서 0이 차지하는 정도로, 비율이 높을수록 저장 공간 사용이 비효율적인 특성
- **유클리드 거리(Euclidean Distance)**: 두 벡터 사이의 직선거리를 나타내는 값
- **`np.count_nonzero()`**: 조건을 만족해 0이 아닌 값으로 표시된 원소의 개수를 세는 함수
- **`np.linalg.norm()`**: 벡터 차이의 크기를 계산해 두 벡터 사이의 거리를 구하는 함수

어휘가 $V$개이면 토큰마다 길이 $V$의 벡터가 필요해 차원이 어휘 크기와 함께 증가한다. 서로 다른 두 원-핫 벡터의 차이에는 `1`과 `-1`이 하나씩 있으므로 거리는 의미와 관계없이 $\sqrt{1^2+(-1)^2}=\sqrt{2}$이다.


In [5]:
zero_count = np.count_nonzero(one_hot_vectors == 0)
zero_ratio = zero_count / one_hot_vectors.size

love_vector = np.eye(vocab_size, dtype=np.int64)[word_to_index["love"]]
nlp_vector = np.eye(vocab_size, dtype=np.int64)[word_to_index["NLP"]]
i_vector = np.eye(vocab_size, dtype=np.int64)[word_to_index["I"]]

different_word_distance = np.linalg.norm(love_vector - nlp_vector)
same_word_distance = np.linalg.norm(i_vector - i_vector)

print(f"0인 원소: {zero_count}/{one_hot_vectors.size} ({zero_ratio:.2%})")
print(f"love ↔ NLP 거리: {different_word_distance:.4f}")
print(f"I ↔ I 거리: {same_word_distance:.1f}")


0인 원소: 54/63 (85.71%)
love ↔ NLP 거리: 1.4142
I ↔ I 거리: 0.0


## 02. 여러 문장의 원-핫 인코딩

여러 문장을 묶으려면 문장별 ID 목록의 길이를 패딩으로 맞춰야 한다.

### 예제 문서 준비

어린왕자 소개문의 문장 순서를 유지해 여러 분류 샘플로 사용한다.


In [6]:
raw_text = """The Little Prince, written by Antoine de Saint-Exupéry, is a poetic tale about a young prince who travels from his home planet to Earth. The story begins with a pilot stranded in the Sahara Desert after his plane crashes. While trying to fix his plane, he meets a mysterious young boy, the Little Prince.

The Little Prince comes from a small asteroid called B-612, where he lives alone with a rose that he loves deeply. He recounts his journey to the pilot, describing his visits to several other planets. Each planet is inhabited by a different character, such as a king, a vain man, a drunkard, a businessman, a geographer, and a fox. Through these encounters, the Prince learns valuable lessons about love, responsibility, and the nature of adult behavior.

On Earth, the Little Prince meets various creatures, including a fox, who teaches him about relationships and the importance of taming, which means building ties with others. The fox's famous line, "You become responsible, forever, for what you have tamed," resonates with the Prince's feelings for his rose.

Ultimately, the Little Prince realizes that the essence of life is often invisible and can only be seen with the heart. After sharing his wisdom with the pilot, he prepares to return to his asteroid and his beloved rose. The story concludes with the pilot reflecting on the lessons learned from the Little Prince and the enduring impact of their friendship.

The narrative is a beautifully simple yet profound exploration of love, loss, and the importance of seeing beyond the surface of things."""

print("문자 수:", len(raw_text))
print("문단 수:", len(raw_text.split("\n\n")))


문자 수: 1567
문단 수: 5


### 문장별 토큰 정규화

문장을 토큰화한 뒤 대소문자를 통일하고 영문자만 남긴다. 불용어와 짧은 토큰 제거는 원-핫 변환이 아니라 단어 집합을 준비하는 선행 단계이다.

- **`sent_tokenize()`**: 문서 문자열을 문장 문자열 목록으로 나누는 함수
- **`word_tokenize()`**: 문장 문자열을 단어와 구두점 단위의 토큰 목록으로 나누는 함수
- **`casefold()`**: 대소문자 차이를 줄이도록 문자열을 소문자 중심으로 정규화하는 메서드
- **`isalpha()`**: 문자열이 문자로만 이루어졌는지 확인하는 메서드
- **불용어(Stopword)**: 문장에 자주 나타나지만 현재 분석에서 구별 능력이 낮아 제거하는 단어


In [7]:
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize

sentences = sent_tokenize(raw_text)

english_stopwords = set(stopwords.words("english"))
preprocessed_sentences = []

for sentence_text in sentences:

    normalized_tokens = [
        token.casefold()
        for token in word_tokenize(sentence_text)
    ]

    filtered_tokens = [
        token
        for token in normalized_tokens
        if token.isalpha()
        and token not in english_stopwords
        and len(token) > 2
    ]
    preprocessed_sentences.append(filtered_tokens)

print("문장 수:", len(preprocessed_sentences))
print("첫 문장 토큰:", preprocessed_sentences[0])
print("첫 문장 토큰 수:", len(preprocessed_sentences[0]))


문장 수: 13
첫 문장 토큰: ['little', 'prince', 'written', 'antoine', 'poetic', 'tale', 'young', 'prince', 'travels', 'home', 'planet', 'earth']
첫 문장 토큰 수: 12


### 제한된 단어 집합과 패딩

빈도 상위 14개 토큰에 두 특수 토큰을 더해 ID 0~15를 만든다. 각 문장은 최대 16개 ID만 남기고 앞쪽을 패딩한다.

- **`<PAD>`**: 길이가 짧은 문장의 빈 위치를 채우는 패딩 토큰
- **`<OOV>`**: 단어 집합에 등록되지 않은 토큰을 대신하는 토큰
- **패딩(Padding)**: 길이가 다른 ID 목록을 같은 길이로 맞추는 과정
- **절단(Truncation)**: 정해진 최대 길이를 넘는 ID를 잘라내는 과정
- **`most_common()`**: `Counter`에서 등장 빈도가 높은 항목을 원하는 개수만큼 반환하는 메서드
- **`dict.get()`**: 키가 없을 때 오류 대신 지정한 기본값을 반환하는 사전 메서드
- **`np.full()`**: 지정한 shape의 모든 원소를 같은 값으로 채운 NumPy 배열을 만드는 함수

정제된 토큰은 단어 집합을 거쳐 ID가 되고, OOV 처리와 절단·패딩을 거치면 모든 문장이 길이 16인 배열로 정렬된다.


In [8]:
# 13문장에 존재하는 모든 토큰 하나로 합치기
all_tokens = [
    token
    for sentence_tokens in preprocessed_sentences
    for token in sentence_tokens
]

# 같은 토큰 개수 세기
corpus_counts = Counter(all_tokens)

PAD_TOKEN = "<PAD>"
OOV_TOKEN = "<OOV>"
MAX_KNOWN_TOKENS = 14
MAX_LENGTH = 16

# 빈도 상위 토큰에 2부터 IOD 부여 -> 단어-토큰ID 정방향 사전 (모델 학습용)
word_to_id = {PAD_TOKEN: 0, OOV_TOKEN: 1}
for token_id, (word, _) in enumerate(
    corpus_counts.most_common(MAX_KNOWN_TOKENS),  # 빈도수 내림차순
    start=2,
):
    word_to_id[word] = token_id

# 단어-토큰ID 역방향 사전 (해설용)
id_to_word = {
    token_id: word
    for word, token_id in word_to_id.items()
}

# 사전에 없는 토큰을 OOV_TOKEN == 1로 변환
integer_sequences = [
    [
        word_to_id.get(token, word_to_id[OOV_TOKEN])
        for token in sentence_tokens
    ]
    for sentence_tokens in preprocessed_sentences
]

# 0으로 가득 채운 (문장 수(13), MAX_LENGTH(16)) 배열 생성
padded_sequences = np.full(
    (len(integer_sequences), MAX_LENGTH),
    fill_value=word_to_id[PAD_TOKEN],
    dtype=np.int64,
)

# 각 행을 padded_sequences 배열에 복사 (오른쪽 정렬, 왼쪽은 0으로 채움)
for row_index, token_ids in enumerate(integer_sequences):
    truncated_ids = token_ids[:MAX_LENGTH]
    padded_sequences[row_index, -len(truncated_ids):] = truncated_ids

first_decoded = [
    id_to_word[token_id]
    for token_id in padded_sequences[0]
]

print("단어 집합:", word_to_id)
print("패딩 배열 shape:", padded_sequences.shape)
print("첫 문장 ID:", padded_sequences[0].tolist())
print("첫 문장 복원:", first_decoded)


단어 집합: {'<PAD>': 0, '<OOV>': 1, 'prince': 2, 'little': 3, 'pilot': 4, 'rose': 5, 'fox': 6, 'young': 7, 'planet': 8, 'earth': 9, 'story': 10, 'plane': 11, 'meets': 12, 'asteroid': 13, 'lessons': 14, 'love': 15}
패딩 배열 shape: (13, 16)
첫 문장 ID: [0, 0, 0, 0, 3, 2, 1, 1, 1, 1, 7, 2, 1, 1, 8, 9]
첫 문장 복원: ['<PAD>', '<PAD>', '<PAD>', '<PAD>', 'little', 'prince', '<OOV>', '<OOV>', '<OOV>', '<OOV>', 'young', 'prince', '<OOV>', '<OOV>', 'planet', 'earth']


### 패딩된 문장을 PyTorch 원-핫 텐서로 변환하기

앞에서 만든 `padded_sequences`는 13개 문장을 길이 16으로 맞춘 정수 ID 배열이다. 이 배열을 PyTorch 모델의 입력으로 사용하려면 먼저 Tensor로 바꾸고, 각 토큰 ID를 길이 16의 원-핫 벡터로 변환한다.

```text
정수 ID 배열                 원-핫 텐서
(13, 16)       →          (13, 16, 16)
 문장  토큰 위치             문장  토큰 위치  단어 ID
```

- **Tensor**: PyTorch가 다차원 수치 데이터를 표현하는 자료구조
- **`torch.tensor()`**: NumPy 배열을 PyTorch Tensor로 변환하는 함수
- **`F.one_hot()`**: 정수 ID를 단어 집합 크기의 원-핫 벡터로 변환하는 함수
- **`to(torch.float32)`**: 정수 원-핫 값을 신경망이 계산할 수 있는 실수 자료형으로 변환하는 메서드
- **패딩 마스크(Padding Mask)**: PAD가 들어 있는 토큰 위치를 `True`로 표시한 불리언 Tensor
- **`eq()`**: Tensor의 각 값이 지정한 값과 같은지 비교하는 메서드

`X[0]`은 첫 번째 문장 전체, `X[0][4]`는 첫 번째 문장의 토큰 위치 인덱스 4에 있는 원-핫 벡터이다. 이 위치의 토큰은 ID가 3인 `little`이므로 `X[0][4]`에서는 인덱스 3만 1이다.

`F.one_hot()`은 PAD ID 0도 `[1, 0, ..., 0]`으로 변환한다. 이번 예제에서는 PAD를 단어 ID 0이라는 특성으로 사용하지 않도록 `padding_mask`가 선택한 벡터 전체를 0으로 바꾼다. 따라서 `X[0].sum(dim=1)`은 첫 문장의 토큰 위치별 원-핫 합을 반환하며, PAD 위치는 0이고 나머지 토큰 위치는 1이다.


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# torch 모델에 사용될 수 있도록 tensor로 변경
# dtype=torch.long -> 다중 분류 손실 함수(CrossEntropyLoss)를 이용하려면 long type(== np.int64) 이어야 한다
token_ids = torch.tensor(padded_sequences, dtype=torch.long)

# 정답(class)의 개수
num_classes = len(word_to_id)

# One-Hot 인코딩 수행
X = F.one_hot(token_ids, num_classes=num_classes).to(torch.float32)

print("정수 ID shape: ", token_ids.shape)  # (13, 16)
print("원-핫 shape: ", X.shape)  # (13, 16, 16)
# print(X[0][0])

# PAD 위치(0번 인덱스)도 원-핫 인코딩으로 1로 변환되기 때문에 다시 0으로 변경 -> 1로 되어 있으면 PAD도 학습을 진행하기 때문!
padding_mask = token_ids.eq(word_to_id[PAD_TOKEN])
X[padding_mask] = 0.0


print("정수 ID shape:", tuple(token_ids.shape))
print("원-핫 shape:", tuple(X.shape))
print("첫 문장 ID:", token_ids[0].tolist())
print("첫 문장 0번 토큰 벡터:", X[0][0].tolist())

print("첫 문장 4번 토큰 벡터:", X[0][4].tolist())
# 4번 토큰의 원-핫 벡터를 확인해보면 3번 인덱스에 1이 작성되어있음
# == 4번 토큰 자리에 작성된 단어는 단어 집합 3번인 'little'이다

print("첫 문장의 각 토큰 위치별 원-핫 벡터 합:", X[0].sum(dim=1).tolist())

정수 ID shape:  torch.Size([13, 16])
원-핫 shape:  torch.Size([13, 16, 16])
정수 ID shape: (13, 16)
원-핫 shape: (13, 16, 16)
첫 문장 ID: [0, 0, 0, 0, 3, 2, 1, 1, 1, 1, 7, 2, 1, 1, 8, 9]
첫 문장 0번 토큰 벡터: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
첫 문장 4번 토큰 벡터: [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
첫 문장의 각 토큰 위치별 원-핫 벡터 합: [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


### 토큰 위치·단어 ID·활성 열 연결하기

첫 번째 문장의 각 토큰 위치를 순회하면서 그 위치의 토큰, 단어 ID와 원-핫 벡터에서 값이 1인 인덱스를 함께 출력한다.

- **토큰 위치 인덱스**: 패딩된 문장 안에서 토큰이 들어 있는 순서 번호
- **단어 ID**: 단어 집합에서 토큰을 구분하기 위해 부여한 정수
- **활성 열(Active Column)**: 원-핫 벡터에서 값이 1인 인덱스
- **`item()`**: 원소가 하나인 Tensor에서 Python 숫자를 꺼내는 메서드
- **`torch.nonzero()`**: 값이 0이 아닌 원소의 인덱스를 반환하는 함수
- **`flatten()`**: 여러 축을 가진 Tensor를 1차원으로 펼치는 메서드

예를 들어 `위치 4 | 토큰=little | ID=3 | 1의 열=[3]`은 첫 문장의 토큰 위치 인덱스 4에 `little`이 있고, 이 단어의 ID와 원-핫 벡터에서 1인 인덱스가 모두 3이라는 의미이다.

위치 0~3의 `<PAD>`는 벡터 전체를 0으로 바꿨으므로 활성 열이 `[]`로 출력된다. PAD가 아닌 토큰은 `ID`와 `1의 열`이 같은 값 하나로 출력되어야 정상이다.


In [15]:
for position, (token_id, vector) in enumerate(zip(token_ids[0], X[0])):

    token_id_value = token_id.item()

    active_columns = torch.nonzero(vector).flatten().tolist()
    token = id_to_word[token_id_value]

    print(
        f"위치 {position:2d} | 토큰={token:>6} | "
        f"ID={token_id_value:2d} | 1의 열={active_columns}"
    )


위치  0 | 토큰= <PAD> | ID= 0 | 1의 열=[]
위치  1 | 토큰= <PAD> | ID= 0 | 1의 열=[]
위치  2 | 토큰= <PAD> | ID= 0 | 1의 열=[]
위치  3 | 토큰= <PAD> | ID= 0 | 1의 열=[]
위치  4 | 토큰=little | ID= 3 | 1의 열=[3]
위치  5 | 토큰=prince | ID= 2 | 1의 열=[2]
위치  6 | 토큰= <OOV> | ID= 1 | 1의 열=[1]
위치  7 | 토큰= <OOV> | ID= 1 | 1의 열=[1]
위치  8 | 토큰= <OOV> | ID= 1 | 1의 열=[1]
위치  9 | 토큰= <OOV> | ID= 1 | 1의 열=[1]
위치 10 | 토큰= young | ID= 7 | 1의 열=[7]
위치 11 | 토큰=prince | ID= 2 | 1의 열=[2]
위치 12 | 토큰= <OOV> | ID= 1 | 1의 열=[1]
위치 13 | 토큰= <OOV> | ID= 1 | 1의 열=[1]
위치 14 | 토큰=planet | ID= 8 | 1의 열=[8]
위치 15 | 토큰= earth | ID= 9 | 1의 열=[9]


### 원-핫 입력의 두 가지 한계

이 코드는 원-핫 표현의 공간 사용량과 단어 사이의 거리를 계산하여 실제 NLP에서 임베딩을 주로 사용하는 이유를 확인한다.

#### 대부분의 값이 0인 희소 표현

`X`의 shape는 `(13, 16, 16)`이므로 전체 원소 수는 `13 × 16 × 16 = 3,328`개이다. PAD가 아닌 토큰 위치는 137개이고 각 원-핫 벡터에는 1이 하나만 있으므로, 0이 아닌 원소도 137개뿐이다.

```text
0이 아닌 원소: 137개
0인 원소: 3,328 - 137 = 3,191개
0의 비율: 3,191 / 3,328 = 95.88%
```

단어 집합이 커지면 원-핫 벡터의 길이도 함께 증가하지만 실제로 사용하는 값은 하나뿐이어서 저장 공간과 연산이 비효율적이다.

#### 단어의 의미 관계를 표현하지 못하는 거리

`prince`와 `little`은 서로 다른 인덱스 하나씩만 1이므로 두 벡터의 유클리드 거리는 `1.4142 = √2`이다. 원-핫에서는 의미가 가까운 단어와 관련 없는 단어 모두 서로 다르기만 하면 거리가 `√2`이므로 단어의 의미 유사성을 구분할 수 없다.

- **희소성(Sparsity)**: 전체 원소 중 0이 차지하는 정도
- **`torch.count_nonzero()`**: Tensor에서 0이 아닌 원소의 개수를 세는 함수
- **`numel()`**: Tensor의 전체 원소 수를 반환하는 메서드
- **`torch.linalg.vector_norm()`**: 두 벡터 차이의 크기를 계산하는 함수

원-핫 벡터는 단어를 서로 다른 ID로 구분할 수 있지만 공간을 많이 사용하고 의미 관계를 표현하지 못한다. 이러한 한계를 줄이기 위해 큰 단어 집합을 사용하는 NLP 모델에서는 임베딩을 주로 사용한다.

In [16]:
non_zero_count = torch.count_nonzero(X).item()
total_count = X.numel()
zero_ratio = 1 - (non_zero_count / total_count)

prince_vector = F.one_hot(
    torch.tensor(word_to_id["prince"]),
    num_classes=num_classes,
).to(torch.float32)
little_vector = F.one_hot(
    torch.tensor(word_to_id["little"]),
    num_classes=num_classes,
).to(torch.float32)

word_distance = torch.linalg.vector_norm(prince_vector - little_vector).item()

print(f"0이 아닌 원소: {non_zero_count}/{total_count}")
print(f"0의 비율: {zero_ratio:.2%}")
print(f"prince ↔ little 거리: {word_distance:.4f}")


0이 아닌 원소: 137/3328
0의 비율: 95.88%
prince ↔ little 거리: 1.4142


## 03. 원-핫 문장 입력과 정답 레이블 준비

앞에서 만든 `X`는 13개 문장을 `(문장 수, 토큰 위치 수, 단어 집합 크기)`로 표현한 모델 입력이다. 지도학습으로 문장을 분류하려면 `X`의 각 문장에 모델이 맞혀야 하는 정답 레이블 `y`를 하나씩 대응시켜야 한다.

- **문장 분류(Sentence Classification)**: 문장마다 미리 정한 클래스 중 하나를 예측하는 작업
- **지도학습(Supervised Learning)**: 입력과 정답 레이블의 관계를 학습하는 방식
- **레이블(Label)**: 모델이 예측해야 하는 정답 클래스
- **`label_to_id`**: 클래스 이름을 모델이 계산할 정수 ID로 바꾸는 사전
- **`id_to_label`**: 예측한 정수 ID를 사람이 읽을 클래스 이름으로 되돌리는 사전

이번 예제에서는 수업 흐름을 확인하기 위해 각 문장에 다음 클래스를 직접 지정한다.

- **SUMMARY**: 이야기의 전체 내용을 설명하는 문장
- **CHARACTER**: 등장인물이나 대상의 특징을 설명하는 문장
- **MESSAGE**: 작품의 교훈이나 의미를 설명하는 문장

`SUMMARY=0`, `CHARACTER=1`, `MESSAGE=2`로 변환한 13개 정답 ID를 `y`에 저장한다. 입력 `X`의 첫 번째 축과 정답 `y`의 길이는 모두 13이어야 각 문장에 정답 하나가 대응한다.

출력에서는 클래스별 문장 수가 각각 5개, 2개, 6개로 나타난다. 이 레이블은 분류 흐름을 보여 주기 위해 직접 만든 작은 불균형 데이터이므로 학습 결과를 실제 문장 분류 성능으로 해석하지 않는다.

In [17]:
# 클래스(정답) 정보 dict 생성
label_to_id = {
    "SUMMARY": 0,
    "CHARACTER": 1,
    "MESSAGE": 2,
}

# 역방향 사전 생성
id_to_label = {
    label_id: label_name
    for label_name, label_id in label_to_id.items()
}

# 학습에 사용할 문장별 정답을 직접 작성
labels = [0, 0, 0, 1, 0, 1, 2, 2, 2, 2, 0, 2, 2]

# pytorch 딥러닝 모델 학습 시 데이터 타입은 무조건 Tensor!
y = torch.tensor(labels, dtype=torch.long)

print("입력 문장 수:", X.shape[0])
print("정답 shape:", tuple(y.shape))
print("클래스별 문장 수:", Counter(labels))
print("레이블 사전:", label_to_id)


입력 문장 수: 13
정답 shape: (13,)
클래스별 문장 수: Counter({2: 6, 0: 5, 1: 2})
레이블 사전: {'SUMMARY': 0, 'CHARACTER': 1, 'MESSAGE': 2}


### RNN 분류기와 학습 도구 준비

이 코드는 원-핫 텐서 `X`가 문장 분류 모델의 입력으로 전달되는 과정을 확인하기 위한 작은 RNN을 만든다. RNN은 각 문장의 토큰 위치를 순서대로 읽고 마지막 은닉 상태에 문장 정보를 요약하며, `nn.Linear`는 이를 세 클래스의 로짓으로 변환한다.

- **`nn.RNN`**: 토큰을 순서대로 처리하며 이전 위치까지의 정보를 은닉 상태에 누적하는 계층
- **`batch_first=True`**: 입력 축을 `(문장 수, 토큰 위치 수, 입력 특성 수)` 순서로 사용하는 설정
- **은닉 상태(Hidden State)**: RNN이 현재 위치까지 읽은 토큰 정보를 요약한 벡터
- **`nn.Linear`**: 마지막 은닉 상태를 클래스별 점수로 변환하는 계층
- **로짓(Logit)**: 확률로 변환하기 전의 클래스별 원시 예측 점수

```text
원-핫 입력 X      마지막 은닉 상태       클래스 로짓
(13, 16, 16)  →      (13, 16)       →    (13, 3)
```

`hidden_state`의 실제 shape는 `(RNN 층 수, 문장 수, hidden_dim)=(1, 13, 16)`이다. `hidden_state[-1]`은 마지막 RNN 층의 결과만 선택하여 `(13, 16)`으로 만들고, `nn.Linear`가 문장마다 `SUMMARY`, `CHARACTER`, `MESSAGE`에 대한 로짓 세 개를 만든다.

같은 코드셀에서 다음 학습 도구도 준비한다.

- **`nn.CrossEntropyLoss`**: `(문장 수, 클래스 수)` 로짓과 `(문장 수,)` 정답 ID를 비교해 분류 손실을 계산하는 함수
- **`optim.Adam`**: 손실 기울기를 이용해 모델 파라미터를 갱신하는 최적화 알고리즘
- **`torch.no_grad()`**: 기울기 기록 없이 모델을 실행하는 문맥 관리자

`CrossEntropyLoss`는 내부에서 로짓을 확률 계산에 알맞게 처리하므로 모델 출력에 소프트맥스를 미리 적용하지 않는다. `torch.no_grad()`로 만든 `initial_logits`는 학습 전 모델이 `(13, 3)` 로짓을 정상적으로 출력하는지 shape만 확인하는 값이다.


In [18]:
import torch.optim as optim

torch.manual_seed(42)

class RNNTextClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()

        # 입력 마지막 축 16을 순서대로 읽어 길이 16의 은닉 상태로 요약
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )

        # 마지막 은닉 상태를 클래스별 로짓으로 변경
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, inputs):
        # hidden_state shape은 `(층 수, 문장 수, hidden_dim)`이다.
        _, hidden_state = self.rnn(inputs)

        # 마지막 층의 shape `(문장 수, hidden_dim)`만 분류기에 전달한다.
        final_hidden = hidden_state[-1]
        return self.classifier(final_hidden)


# input_dim은 원-핫 마지막 축, 출력 수는 클래스 수로 결정
model = RNNTextClassifier(
    input_dim=X.shape[2],
    hidden_dim=16,
    num_classes=len(label_to_id),
)

# 손실은 `(문장 수, 클래스 수)` 로짓과 `(문장 수,)` 정수 정답을 비교
criterion = nn.CrossEntropyLoss()

# Adam이 손실 기울기로 모델 파라미터를 갱신
optimizer = optim.Adam(model.parameters(), lr=0.005)

with torch.no_grad():
    initial_logits = model(X)

print("모델 입력 shape:", tuple(X.shape))
print("초기 로짓 shape:", tuple(initial_logits.shape))

모델 입력 shape: (13, 16, 16)
초기 로짓 shape: (13, 3)


### 손실을 줄이도록 RNN 학습하기

모델은 같은 13개 문장을 100번 반복해서 읽으며 예측 로짓과 정답 `y`의 차이를 줄인다. 전체 데이터를 한 번 학습하는 단위를 epoch라고 하며, 각 epoch에서는 다음 순서를 반복한다.

```text
기울기 초기화 → 순전파 → 손실 계산 → 역전파 → 파라미터 갱신
zero_grad()      model(X)   criterion()   backward()    step()
```

- **Epoch**: 전체 학습 데이터를 한 번 사용해 학습하는 단위
- **순전파(Forward Propagation)**: 입력을 모델에 통과시켜 예측 로짓을 계산하는 과정
- **손실(Loss)**: 예측 로짓과 정답의 차이를 하나의 값으로 나타낸 학습 기준
- **역전파(Backpropagation)**: 손실을 줄이기 위해 각 파라미터가 어느 방향으로 변해야 하는지 기울기를 계산하는 과정
- **`zero_grad()`**: 이전 epoch에서 남은 기울기를 초기화하는 메서드
- **`backward()`**: 현재 손실을 기준으로 모델 파라미터의 기울기를 계산하는 메서드
- **`step()`**: 계산된 기울기를 사용해 optimizer가 파라미터를 갱신하게 하는 메서드

출력에서 손실은 1 epoch의 `1.1597`에서 100 epoch의 `0.0083`으로 감소한다. 이는 모델이 학습에 사용한 13개 문장의 정답을 점점 잘 맞추고 있다는 의미일 뿐, 새로운 문장도 잘 분류한다는 것을 의미하지 않는다.


In [19]:
loss_history = []

for epoch in range(1, 101):
    # 이전 epoch의 누적 기울기를 지운다.
    optimizer.zero_grad()

    # 순전파 로짓과 정답으로 손실을 계산한다.
    output = model(X)
    loss = criterion(output, y)

    # 역전파로 기울기를 구하고 파라미터를 갱신한다.
    loss.backward()
    optimizer.step()

    # Python 실수 손실을 저장한다.
    loss_history.append(loss.item())

    if epoch == 1 or epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss {loss.item():.4f}")

Epoch   1 | Loss 1.1597
Epoch  20 | Loss 0.9371
Epoch  40 | Loss 0.4599
Epoch  60 | Loss 0.1362
Epoch  80 | Loss 0.0275
Epoch 100 | Loss 0.0083


### 학습한 문장의 예측 결과 확인하기

학습이 끝난 모델에 같은 13개 문장 `X`를 다시 입력하고, 각 문장의 세 로짓 중 가장 큰 값의 인덱스를 예측 ID로 선택한다. 예측 ID를 `id_to_label`로 변환하면 `SUMMARY`, `CHARACTER`, `MESSAGE`라는 클래스 이름으로 결과를 읽을 수 있다.

- **`eval()`**: 모델을 평가 모드로 전환하는 메서드
- **`torch.no_grad()`**: 예측 중 기울기 기록을 중단하여 불필요한 메모리 사용을 줄이는 문맥 관리자
- **`argmax(dim=1)`**: 각 문장의 클래스 축에서 가장 큰 로짓이 있는 인덱스를 반환하는 메서드
- **`eq()`**: 예측 ID와 정답 ID가 같은지 원소별로 비교하는 메서드
- **정확도(Accuracy)**: 전체 예측 중 정답과 일치한 예측의 비율

로짓에 소프트맥스를 적용해도 가장 큰 값의 위치는 변하지 않으므로 클래스 ID만 고를 때는 로짓에 바로 `argmax()`를 사용할 수 있다.

출력에서는 13개의 예측 ID가 정답 ID와 모두 같아 학습 데이터 정확도가 `100.00%`로 나타난다. 그러나 학습에 사용한 문장을 다시 예측한 결과이므로 모델이 해당 13개 문장을 외웠을 가능성이 있다. 보지 못한 문장에 대한 분류 성능은 별도의 검증 데이터나 테스트 데이터로 확인해야 한다.


In [20]:
model.eval()  # 평가 모드

with torch.no_grad():  # 기울기 기록 X
    logits = model(X)  # 모델에 값 입력 후 결과 logit 반환 (13, 3)

    predictions = logits.argmax(dim=1)
    # 각 행 마다 logit 값이 가장 높은 열의 인덱스 구하기 (0, 1, 2 중 하나) * 13 문장

    train_accuracy = predictions.eq(y).to(torch.float32).mean().item()

print("실제 ID:", y.tolist())
print("예측 ID:", predictions.tolist())
print("실제 이름:", [id_to_label[label_id] for label_id in y.tolist()])
print("예측 이름:", [id_to_label[label_id] for label_id in predictions.tolist()])
print(f"학습 데이터 정확도: {train_accuracy:.2%}")


실제 ID: [0, 0, 0, 1, 0, 1, 2, 2, 2, 2, 0, 2, 2]
예측 ID: [0, 0, 0, 1, 0, 1, 2, 2, 2, 2, 0, 2, 2]
실제 이름: ['SUMMARY', 'SUMMARY', 'SUMMARY', 'CHARACTER', 'SUMMARY', 'CHARACTER', 'MESSAGE', 'MESSAGE', 'MESSAGE', 'MESSAGE', 'SUMMARY', 'MESSAGE', 'MESSAGE']
예측 이름: ['SUMMARY', 'SUMMARY', 'SUMMARY', 'CHARACTER', 'SUMMARY', 'CHARACTER', 'MESSAGE', 'MESSAGE', 'MESSAGE', 'MESSAGE', 'SUMMARY', 'MESSAGE', 'MESSAGE']
학습 데이터 정확도: 100.00%
